# Graph Convolutions & GraphSAGE

Companion notebook for the [Graph Convolutions lesson](https://ml-viz-ruby.vercel.app/courses/graph-neural-networks/02-graph-convolutions).

We implement a **GCN layer** from scratch in NumPy — including the symmetric normalization
`Â = D̃^{-1/2}(A+I)D̃^{-1/2}` — run a 2-layer forward pass, and compare GCN's full-neighborhood
aggregation to **GraphSAGE-style neighbor sampling**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — The normalized adjacency Â

Add self-loops, then symmetrically normalize by degree. The √ on both sides down-weights
high-degree neighbors (a hub shouldn't dominate).

In [ ]:
def normalized_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])          # add self-loops
    d = A_tilde.sum(axis=1)                    # degrees
    D_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt   # symmetric normalization

A = np.zeros((5, 5))
for u, v in [(0,1),(1,2),(2,3),(3,4),(1,3)]:
    A[u,v] = A[v,u] = 1
A_hat = normalized_adjacency(A)
print('Â =\n', A_hat)
print('row sums (not 1 — symmetric, not row-stochastic):', A_hat.sum(1))

### The lesson's 3-node worked example

Reproduce the path graph $1\!-\!2\!-\!3$ trace from the lesson: self-loop entries
$0.5, 0.333, 0.5$, edge entries $1/\sqrt{6} \approx 0.408$, and one aggregation of
$\mathbf{h} = [1, 0, -1]$ giving $[0.5, 0, -0.5]$.

In [ ]:
A_path = np.array([[0,1,0],[1,0,1],[0,1,0]], float)
A_hat_path = normalized_adjacency(A_path)
h = np.array([1.0, 0.0, -1.0])
print('Â =\n', A_hat_path.round(3))
print('one aggregation Âh =', (A_hat_path @ h).round(3))   # → [ 0.5  0. -0.5]
print('one more layer Â(Âh) =', (A_hat_path @ (A_hat_path @ h)).round(3))


## 2 — A GCN layer and a 2-layer forward pass

One layer is `σ(Â H W)`. We stack two layers (hidden ReLU, then a linear output) — a complete GCN
forward pass for, say, 3-class node classification.

In [ ]:
def relu(x):
    return np.maximum(x, 0)

def gcn_layer(A_hat, H, W):
    return A_hat @ H @ W

n, d_in, d_hid, d_out = 5, 4, 8, 3
X = rng.normal(size=(n, d_in))                 # node features
W1 = rng.normal(size=(d_in, d_hid)) * 0.5
W2 = rng.normal(size=(d_hid, d_out)) * 0.5

H1 = relu(gcn_layer(A_hat, X, W1))             # layer 1 (2-hop info after layer 2)
logits = gcn_layer(A_hat, H1, W2)              # layer 2 -> class logits per node
probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
print('per-node class probabilities:\n', probs)
print('predicted classes:', probs.argmax(1))

## 3 — GraphSAGE: sampling neighbors

On a huge graph you can't aggregate every neighbor of a hub. GraphSAGE samples a fixed number per
node. We compare the full mean aggregation to a sampled estimate — sampling approximates the full
aggregation while bounding cost regardless of degree.

In [ ]:
def full_mean(A, X, v):
    nb = np.where(A[v] > 0)[0]
    return X[nb].mean(0)

def sampled_mean(A, X, v, k, rng):
    nb = np.where(A[v] > 0)[0]
    take = rng.choice(nb, size=min(k, len(nb)), replace=False)
    return X[take].mean(0)

# make node 1 a hub with many neighbors
Abig = A.copy()
Xbig = rng.normal(size=(5, 4))
node = 1
full = full_mean(Abig, Xbig, node)
samp = np.mean([sampled_mean(Abig, Xbig, node, k=2, rng=np.random.default_rng(s)) for s in range(200)], axis=0)
print('full neighborhood mean:   ', full)
print('avg of sampled estimates: ', samp)
print('sampling is an unbiased estimate of the full mean ->', np.allclose(full, samp, atol=0.05))

## ✏️ Your turn

**Exercise.** Implement `gcn_forward(A, X, W1, W2)` doing a full 2-layer GCN forward pass: build the
normalized adjacency, apply `relu(Â X W1)`, then `Â H1 W2`, and return the logits. Reuse
`normalized_adjacency`, `relu`, and `gcn_layer` above.

In [ ]:
def gcn_forward(A, X, W1, W2):
    # TODO(you): normalize A, run two GCN layers (ReLU after the first), return logits
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
out = gcn_forward(A, X, W1, W2)
assert out.shape == (5, 3), out.shape
assert np.allclose(out, logits)            # matches the step-by-step pass above
# symmetric normalization is symmetric
assert np.allclose(A_hat, A_hat.T)
print('\u2713 GCN forward pass is correct')

<details>
<summary>Solution</summary>

```python
def gcn_forward(A, X, W1, W2):
    A_hat = normalized_adjacency(A)
    H1 = relu(gcn_layer(A_hat, X, W1))
    return gcn_layer(A_hat, H1, W2)
```

Two layers give each node a 2-hop receptive field. The shared weights W1, W2 are the learnable
'filters' — trained by backprop against node labels, exactly like a CNN, but over graph neighborhoods
instead of pixel grids.

</details>